# Summarize Shifts

Aggregate enriched employee shifts into the canonical yearly summary.

**Requires:** employee `*.enriched.csv` files.  
**Produces:** a CSV or JSON summary and a summary report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("summarize_shifts")
display(nb.pipeline_overview(ctx, "summarize_shifts"))


## Controls


In [ ]:
VERBOSE = True
YEAR_START = int(step_cfg.get("year_start", 2014))
YEAR_END = int(step_cfg.get("year_end", 2025))
OUTPUT_FORMAT = str(step_cfg.get("format", "csv"))
MIN_HOURS = step_cfg.get("min_hours")

{
    "year_start": YEAR_START,
    "year_end": YEAR_END,
    "output_format": OUTPUT_FORMAT,
    "min_hours": MIN_HOURS,
}


## Input Preview


In [ ]:
display(nb.file_table(paths.enrichment_dir, "*.enriched.csv"))
enriched_files = sorted(paths.enrichment_dir.glob("*.enriched.csv"))
if enriched_files:
    display(nb.preview_csv(enriched_files[0]))


## Build Options


In [ ]:
from core.shifts.summary.options import TurniEmployeeSummaryOptions

summary_path = paths.summary_csv if OUTPUT_FORMAT == "csv" else paths.summary_csv.with_suffix(".json")
options = TurniEmployeeSummaryOptions(
    enriched_dir=str(paths.enrichment_dir),
    out=str(summary_path),
    report_json=str(paths.summary_report),
    year_start=YEAR_START,
    year_end=YEAR_END,
    output_format=OUTPUT_FORMAT,
    min_hours=float(MIN_HOURS) if MIN_HOURS is not None else None,
    verbose=VERBOSE,
)
options


## Run Summary


In [ ]:
from core.drive.logging_utils import setup_logging
from core.shifts.summary.service import run_from_options

setup_logging(VERBOSE)
summary_report = run_from_options(options)
display(nb.report_summary(summary_report))


## Inspect Summary


In [ ]:
display(nb.artifact_table({
    "summary": summary_path,
    "summary report": paths.summary_report,
}))
if OUTPUT_FORMAT == "csv":
    display(nb.preview_csv(summary_path, rows=20))
else:
    nb.preview_json(summary_path)
